In [3]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from churnlens.segmentation.features import build_segment_features

df = pd.read_csv("../data/processed/telco_clean.csv")
features = build_segment_features(df)
scaled = StandardScaler().fit_transform(features)
print(features.shape)
features.describe().round(2)

(7043, 6)


,tenure,MonthlyCharges,n_addons,contract_months,internet_dsl,internet_fiber
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,32.37,64.76,2.04,8.84,0.34,0.44
std,24.56,30.09,1.85,9.55,0.47,0.50
min,0.00,18.25,0.00,1.00,0.00,0.00
25%,9.00,35.50,0.00,1.00,0.00,0.00
50%,29.00,70.35,2.00,1.00,0.00,0.00
75%,55.00,89.85,3.00,12.00,1.00,1.00
max,72.00,118.75,6.00,24.00,1.00,1.00


In [4]:
rows = []
for k in range(2,9):
    labelings = [
        KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(scaled)
        for seed in range(5)
    ]
    sil = silhouette_score(scaled, labelings[0])
    stability = np.mean([
        adjusted_rand_score(labelings[0], other) for other in labelings[1:]
    ])
    rows.append({"k":k, "silhouette": round(sil,3), "stability_ARI": round(stability,3)})

pd.DataFrame(rows).set_index("k")

,silhouette,stability_ARI
k,,
2,0.376,1.000
3,0.432,1.000
4,0.438,1.000
5,0.469,1.000
6,0.482,1.000
7,0.449,0.990
8,0.431,0.994


In [5]:
for K in [4, 5, 6]:
    km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(scaled)

    tmp = df.copy()
    tmp["cluster"] = km.labels_
    feats = features.copy()
    feats["cluster"] = km.labels_

    profile = tmp.groupby("cluster").agg(
        customers=("customerID", "count"),
        churn_rate=("Churn", "mean"),
        avg_tenure=("tenure", "mean"),
        avg_monthly=("MonthlyCharges", "mean"),
    ).round(2)
    extra = feats.groupby("cluster").agg(
        avg_addons=("n_addons", "mean"),
        avg_contract_months=("contract_months", "mean"),
        pct_fiber=("internet_fiber", "mean"),
        pct_dsl=("internet_dsl", "mean"),
    ).round(2)

    print(f"=== K = {K} ===")
    print(profile.join(extra).to_string())
    print()

=== K = 4 ===
         customers  churn_rate  avg_tenure  avg_monthly  avg_addons  avg_contract_months  pct_fiber  pct_dsl
cluster                                                                                                     
0             1756        0.25       21.68        52.35        1.95                 4.81       0.00     1.00
1             2378        0.51       23.65        87.59        1.99                 2.18       1.00     0.00
2             1526        0.07       30.55        21.08        0.00                13.24       0.00     0.00
3             1383        0.08       62.95        89.48        4.48                20.53       0.52     0.48

=== K = 5 ===
         customers  churn_rate  avg_tenure  avg_monthly  avg_addons  avg_contract_months  pct_fiber  pct_dsl
cluster                                                                                                     
0             1164        0.18       58.74       102.18        4.05                14.02        1.0

In [ ]:
K = 5
km = KMeans(n_clusters=K, n_init=10, random_state=42).fir(scaled)
df["cluster"] = km.labels_
df["churned_monthly"] = df["MonthlyCharges"] * df["Churn"]

rev = df.groupby("cluster").agg(
    customers=("customerID","count"),
    churn_rate=("Churn", "mean"),
    avg_monthly=("MonthlyCharges", "mean"),
    churned_monthly_revenue=("churned_monthly", "sum"),
)
rev["share_of_customers"] = rev["customers"] / rev["customers"].sum()
rev["share_of_churned_revenue"] = (
    rev["churned_monthly_revenue"] / rev["churned_monthly_revenue"].sum()
)
rev.round(3)